In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path(r"D:\college ml project\EnergySense\data\processed\household_power_cleaned.csv")
df = pd.read_csv(data_path)

# The processed CSV stores timestamp as a column. Convert it and make it the index.
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(
        df['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise'
    )
    df = df.set_index('timestamp')
elif not isinstance(df.index, pd.DatetimeIndex):
    # Fallback for a CSV whose timestamp was already written as the first index column.
    first_col = df.columns[0]
    if str(first_col).startswith('Unnamed:'):
        parsed_index = pd.to_datetime(df.pop(first_col), errors='raise')
        df.index = parsed_index
    else:
        df.index = pd.to_datetime(df.index, format='%Y-%m-%d %H:%M:%S', errors='raise')

df.index.name = 'timestamp'
print('Data shape:', df.shape)
print('Index type:', type(df.index).__name__)
print('First timestamp:', df.index[0])
df.head()

Data shape: (2075259, 9)
Index type: DatetimeIndex
First timestamp: 2006-12-16 17:24:00


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
timestamp,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [2]:
# Verify timestamp handling. Cell 1 no longer tries to parse the row-number index.
assert isinstance(df.index, pd.DatetimeIndex), 'Timestamp index was not created correctly.'
assert df.index.is_monotonic_increasing, 'Timestamps are not in chronological order.'
print(type(df.index))
print(df.index[:5])

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
DatetimeIndex(['2006-12-16 17:24:00', '2006-12-16 17:25:00',
               '2006-12-16 17:26:00', '2006-12-16 17:27:00',
               '2006-12-16 17:28:00'],
              dtype='datetime64[ns]', name='timestamp', freq=None)


In [3]:
hour_extract=df.index.hour
day_of_week_extract=df.index.dayofweek
month_extract=df.index.month
weekend_extract=((day_of_week_extract == 5) | (day_of_week_extract == 6)).astype(int)

In [4]:
weekend_extract

array([1, 1, 1, ..., 0, 0, 0], shape=(2075259,))

In [5]:
df['hour']=hour_extract
df['day']=day_of_week_extract
df['month']=month_extract
df['weekends']=weekend_extract

In [6]:
df

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,day,month,weekends
timestamp,,,,,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,17,5,12,1
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,5,12,1
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,5,12,1
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,5,12,1
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,17,5,12,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010-11-26 20:58:00,26/11/2010,20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0,20,4,11,0
2010-11-26 20:59:00,26/11/2010,20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0,20,4,11,0
2010-11-26 21:00:00,26/11/2010,21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0,21,4,11,0


In [7]:
df['lag_1']=df['Global_active_power'].shift(1)

In [8]:
df['lag_10']=df['Global_active_power'].shift(10)
df['lag_30']=df['Global_active_power'].shift(30)
df['lag_60']=df['Global_active_power'].shift(60)

In [9]:
df['lag_5']=df['Global_active_power'].shift(5)

In [10]:
df

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,day,month,weekends,lag_1,lag_10,lag_30,lag_60,lag_5
timestamp,,,,,,,,,,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,17,5,12,1,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,5,12,1,4.216,NaN,NaN,NaN,NaN
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,5,12,1,5.360,NaN,NaN,NaN,NaN
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,5,12,1,5.374,NaN,NaN,NaN,NaN
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,17,5,12,1,5.388,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010-11-26 20:58:00,26/11/2010,20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0,20,4,11,0,0.946,0.912,0.982,1.526,0.944
2010-11-26 20:59:00,26/11/2010,20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0,20,4,11,0,0.946,0.948,0.958,1.524,0.946
2010-11-26 21:00:00,26/11/2010,21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0,21,4,11,0,0.944,1.198,0.980,1.456,0.946


In [11]:
df['rolling_mean_5'] = df['Global_active_power'].rolling(5).mean()
df['rolling_mean_10'] = df['Global_active_power'].rolling(10).mean()
df['rolling_mean_30'] = df['Global_active_power'].rolling(30).mean()
df['rolling_std_10'] = df['Global_active_power'].rolling(10).std()

In [12]:
df.head(15)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,...,weekends,lag_1,lag_10,lag_30,lag_60,lag_5,rolling_mean_5,rolling_mean_10,rolling_mean_30,rolling_std_10
timestamp,,,,,,,,,,,,,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,17,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,...,1,4.216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,...,1,5.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,...,1,5.374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,17,...,1,5.388,NaN,NaN,NaN,NaN,4.8008,NaN,NaN,NaN
2006-12-16 17:29:00,16/12/2006,17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0,17,...,1,3.666,NaN,NaN,NaN,4.216,4.6616,NaN,NaN,NaN
2006-12-16 17:30:00,16/12/2006,17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0,17,...,1,3.520,NaN,NaN,NaN,5.360,4.3300,NaN,NaN,NaN
2006-12-16 17:31:00,16/12/2006,17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0,17,...,1,3.702,NaN,NaN,NaN,5.374,3.9952,NaN,NaN,NaN
2006-12-16 17:32:00,16/12/2006,17:32:00,3.668,0.510,233.99,15.8,0.0,1.0,17.0,17,...,1,3.700,NaN,NaN,NaN,5.388,3.6512,NaN,NaN,NaN


In [13]:
df['target_10min'] = df['Global_active_power'].shift(-10)

In [14]:
df.head(11)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,...,lag_1,lag_10,lag_30,lag_60,lag_5,rolling_mean_5,rolling_mean_10,rolling_mean_30,rolling_std_10,target_10min
timestamp,,,,,,,,,,,,,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.448
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,...,4.216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.412
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,...,5.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.224
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,...,5.374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.268
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,17,...,5.388,NaN,NaN,NaN,NaN,4.8008,NaN,NaN,NaN,4.054
2006-12-16 17:29:00,16/12/2006,17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0,17,...,3.666,NaN,NaN,NaN,4.216,4.6616,NaN,NaN,NaN,3.384
2006-12-16 17:30:00,16/12/2006,17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0,17,...,3.520,NaN,NaN,NaN,5.360,4.3300,NaN,NaN,NaN,3.270
2006-12-16 17:31:00,16/12/2006,17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0,17,...,3.702,NaN,NaN,NaN,5.374,3.9952,NaN,NaN,NaN,3.430
2006-12-16 17:32:00,16/12/2006,17:32:00,3.668,0.510,233.99,15.8,0.0,1.0,17.0,17,...,3.700,NaN,NaN,NaN,5.388,3.6512,NaN,NaN,NaN,3.266


In [15]:
print(df['lag_1'].isna().sum())
print(df['lag_10'].isna().sum())
print(df['lag_30'].isna().sum())
print(df['lag_60'].isna().sum())
print(df['rolling_mean_5'].isna().sum())
print(df['rolling_mean_10'].isna().sum())
print(df['rolling_mean_30'].isna().sum())
print(df['rolling_std_10'].isna().sum())
print(df['target_10min'].isna().sum())



25904
25913
25933
25963
25971
26056
26376
26056
25913


In [16]:
X_features=['hour', 'day', 'month', 'weekends','lag_1','lag_10','lag_30','lag_60','lag_5','rolling_mean_5','rolling_mean_10','rolling_mean_30','rolling_std_10']

In [17]:
df[X_features].notna().all(axis=1).sum()

np.int64(2048426)

In [18]:
columns_to_check=X_features+['target_10min']

In [19]:
valid_mask = df[columns_to_check].notna().all(axis=1)

In [20]:
print(valid_mask)

timestamp
2006-12-16 17:24:00    False
2006-12-16 17:25:00    False
2006-12-16 17:26:00    False
2006-12-16 17:27:00    False
2006-12-16 17:28:00    False
                       ...  
2010-11-26 20:58:00    False
2010-11-26 20:59:00    False
2010-11-26 21:00:00    False
2010-11-26 21:01:00    False
2010-11-26 21:02:00    False
Length: 2075259, dtype: bool


In [21]:
X_features

['hour',
 'day',
 'month',
 'weekends',
 'lag_1',
 'lag_10',
 'lag_30',
 'lag_60',
 'lag_5',
 'rolling_mean_5',
 'rolling_mean_10',
 'rolling_mean_30',
 'rolling_std_10']

In [22]:
if 'Global_active_power' not in X_features:
    X_features.append('Global_active_power')

print(X_features)
print('Total features:', len(X_features))

['hour', 'day', 'month', 'weekends', 'lag_1', 'lag_10', 'lag_30', 'lag_60', 'lag_5', 'rolling_mean_5', 'rolling_mean_10', 'rolling_mean_30', 'rolling_std_10', 'Global_active_power']
Total features: 14


In [23]:
print(X_features)
print("Total features:", len(X_features))

['hour', 'day', 'month', 'weekends', 'lag_1', 'lag_10', 'lag_30', 'lag_60', 'lag_5', 'rolling_mean_5', 'rolling_mean_10', 'rolling_mean_30', 'rolling_std_10', 'Global_active_power']
Total features: 14


In [24]:

import gc
import numpy as np
import pandas as pd

gc.collect()

# Check that all required features and target exist
columns_to_check = X_features + ['target_10min']

# Keep only rows where all inputs and target are available
valid_mask = df[columns_to_check].notna().all(axis=1)

# Create updated X and y
X = df.loc[valid_mask, X_features].astype(np.float32)
y = df.loc[valid_mask, 'target_10min'].astype(np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Total features:", len(X_features))


X shape: (2048270, 14)
y shape: (2048270,)
Total features: 14


## 11. Leakage-Safe Chronological Train-Test Split

The split is chronological. Training rows are purged when their 10-minute-ahead target would fall at or after the first test timestamp. Test features may still use historical observations from before the test boundary.

In [25]:
import pandas as pd

# Use the first 70% of valid reference timestamps as the split boundary.
split_idx = int(len(X) * 0.70)
split_timestamp = X.index[split_idx]

# Keep training labels strictly earlier than the first test timestamp.
train_mask = (X.index + pd.Timedelta(minutes=10)) < split_timestamp
test_mask = X.index >= split_timestamp

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print('Split timestamp:', split_timestamp)
print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)
print('Feature count:', X_train.shape[1])

last_train_target_time = X_train.index.max() + pd.Timedelta(minutes=10)
print('Last training reference:', X_train.index.max())
print('Last training target time:', last_train_target_time)
print('First testing timestamp:', X_test.index.min())
print('No target overlap:', last_train_target_time < X_test.index.min())

Split timestamp: 2009-09-13 17:13:00
X_train: (1433779, 14)
y_train: (1433779,)
X_test: (614481, 14)
y_test: (614481,)
Feature count: 14
Last training reference: 2009-09-13 17:02:00
Last training target time: 2009-09-13 17:12:00
First testing timestamp: 2009-09-13 17:13:00
No target overlap: True


In [26]:
print('Training index aligned:', X_train.index.equals(y_train.index))
print('Testing index aligned:', X_test.index.equals(y_test.index))
print('Training chronological:', X_train.index.is_monotonic_increasing)
print('Testing chronological:', X_test.index.is_monotonic_increasing)
print('Train ends before test:', X_train.index.max() < X_test.index.min())
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.shape[1] == 14
assert last_train_target_time < X_test.index.min(), '10-minute target leakage detected.'
print('All split validation checks passed.')

Training index aligned: True
Testing index aligned: True
Training chronological: True
Testing chronological: True
Train ends before test: True
All split validation checks passed.


## 12. Save Engineered Features for Model Development

### Objective
Save the finalized 14 input features and their matching 10-minute-ahead target for `04_model_development.ipynb`.

### Why save X and y separately?
- `X_features.csv.gz` stores the 14 model input features.
- `y_target.csv.gz` stores the corresponding `target_10min` labels.
- Both files preserve the timestamp index, keeping rows aligned.
- The complete valid dataset is saved. The leakage-safe chronological split is recreated in the model-development notebook.

Run this cell only after all feature creation, target preparation, and split-validation cells have completed successfully.

In [27]:
from pathlib import Path

save_dir = Path(r"D:\college ml project\EnergySense\data\processed")
save_dir.mkdir(parents=True, exist_ok=True)

# Safety checks before exporting.
assert isinstance(X.index, pd.DatetimeIndex)
assert X.index.equals(y.index), 'X and y timestamps are not aligned.'
assert X.shape[1] == 14, f'Expected 14 features, found {X.shape[1]}.'
assert 'target_10min' not in X.columns

X.to_csv(save_dir / 'X_features.csv.gz', compression='gzip', index=True)
y.to_frame(name='target_10min').to_csv(
    save_dir / 'y_target.csv.gz', compression='gzip', index=True
)

print('Engineered features and target saved successfully!')
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Feature count:', X.shape[1])
print('Saved directory:', save_dir)

Engineered features and target saved successfully!
X shape: (2048270, 14)
y shape: (2048270,)
Feature count: 14
Saved directory: D:\college ml project\EnergySense\data\processed
